## Training notebook

In [ ]:
import gym
import highway_env

from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, BaseCallback

from typing import Callable

### Select operating system

In [ ]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['MACOS', 'WINDOWS', 'UBUNTU_MICRO', 'UBUNTU_DELL'])

In [ ]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "MACOS" else 'C:/Users/pigo/Desktop/HighwayDRL/training_output/'
if os_id.value == 'UBUNTU_MICRO':
    OUTPUT_PATH = '/home/utente1/Desktop/PighettiForneris/HighwayDRL/training_output/'
elif os_id.value == 'UBUNTU_DELL':
    OUTPUT_PATH = '/home/utente1/pighetti/HighwayDRL/training_output'

### Select environment

In [ ]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['highway-v0','dm-env-v0','acc-d<m-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

In [ ]:
total_timesteps = 6e5
# Create and wrap the environment
env = gym.make(env_id.value)
env.configure({
    "training_total_timesteps": total_timesteps
})
# Create a TensorboardCallback to plot sub-rewards

class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0] * self.training_env.get_attr('vehicle')[0].crashed
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

In [ ]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [ ]:
ilr = 3.5e-4
# PPO parameters
model = PPO("MlpPolicy",
        env,
        learning_rate=ilr,
        verbose=1,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

In [ ]:
# Train the agent for n steps
model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, eval_callback])

# Save the trained agentP
model.save(f'{OUTPUT_PATH}/models/ppo_RL_BASELINE_{str(os_id.value)}_{total_timesteps:.1E}_{env_id.value}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

## Continual learning

In [ ]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)

In [ ]:
custom_objects = { 'learning_rate': 3.5e-4 }

model_cl = PPO.load(f"{OUTPUT_PATH}models/ppo_RL_UBUNTU_1.0E+05_multipleO-dm-env-v0_15-09_18-29-27", env=env_cl, custom_objects=custom_objects, tensorboard_log=f"{OUTPUT_PATH}logdir")

In [ ]:
total_timesteps = 2e5

In [ ]:
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_num", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

In [ ]:
new_timesteps = 1e5

model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()], reset_num_timesteps=False)

model_cl.save(f'{OUTPUT_PATH}models/ppo_RL_CL_{str(os_id.value)}_{(total_timesteps + new_timesteps):.1E}_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')